# 03 — Modelagem e Seleção do Modelo

## Objetivo

Este notebook desenvolve e compara modelos de classificação para discriminar clientes classificados como bons e maus pagadores segundo a variável `TARGET`.

Serão avaliados diferentes algoritmos de classificação utilizando pipelines que integram o pré-processamento e o estimador. A comparação será realizada principalmente por meio de validação cruzada sobre o conjunto de treino.

Como a variável-alvo é desbalanceada, a avaliação não será baseada apenas em acurácia. Serão consideradas especialmente as métricas **ROC-AUC** e **PR-AUC (Average Precision)**, além de precision, recall, F1-score e matriz de confusão quando apropriado.

O conjunto de teste permanecerá reservado durante a comparação dos modelos e será utilizado apenas para a avaliação final do modelo selecionado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from sklearn.inspection import permutation_importance
from sklearn.base import clone

RANDOM_STATE = 42

## 1. Preparação dos dados para modelagem

Para manter este notebook independente e reproduzível, as bases originais serão carregadas novamente e a variável-alvo será reconstruída utilizando o mesmo critério definido nos notebooks anteriores.

Será considerado `TARGET = 1` o cliente que apresentou pelo menos uma ocorrência de atraso igual ou superior a 30 dias no histórico observado.

Também serão reaplicadas as transformações determinísticas definidas no pré-processamento: criação das variáveis de idade e tempo de emprego em anos, tratamento da codificação especial de `DAYS_EMPLOYED`, representação da ausência de ocupação e exclusão das variáveis que não serão utilizadas como preditoras.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/raw")

application = pd.read_csv(DATA_DIR / "application_record.csv")
credit = pd.read_csv(DATA_DIR / "credit_record.csv")

# Reconstrução do TARGET
ids_comuns = set(application['ID']).intersection(set(credit['ID']))

credit_comum = credit[
    credit['ID'].isin(ids_comuns)
].copy()

ids_maus_pagadores = set(
    credit_comum.loc[
        credit_comum['STATUS'].isin(['1', '2', '3', '4', '5']),
        'ID'
    ]
)

target = pd.DataFrame({'ID': list(ids_comuns)})
target['TARGET'] = target['ID'].isin(ids_maus_pagadores).astype(int)

df = application.merge(target, on='ID', how='inner')

# Transformações determinísticas
df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365.25

df['YEARS_EMPLOYED'] = np.where(
    df['DAYS_EMPLOYED'] == 365243,
    np.nan,
    -df['DAYS_EMPLOYED'] / 365.25
)

df['OCCUPATION_TYPE'] = (
    df['OCCUPATION_TYPE'].fillna('Unknown')
)

df = df.drop(
    columns=['DAYS_BIRTH', 'DAYS_EMPLOYED']
)

# Preditores e variável-alvo
X = df.drop(
    columns=['ID', 'FLAG_MOBIL', 'TARGET']
)

y = df['TARGET']

print(f'Dimensão de X: {X.shape}')
print(f'Dimensão de y: {y.shape}')

print('\nDistribuição do TARGET:')
print(y.value_counts())

## 2. Separação entre treino e teste

Será utilizada a mesma estratégia definida no pré-processamento: **80% dos registros para treino e 20% para teste**, com estratificação pela variável `TARGET`.

O conjunto de treino será utilizado para desenvolvimento, validação cruzada e comparação dos algoritmos. O conjunto de teste não participará da escolha do modelo e permanecerá reservado para a avaliação final.

Essa separação reduz o risco de obter uma estimativa excessivamente otimista do desempenho ao utilizar repetidamente os dados de teste durante o desenvolvimento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')

print('\nTARGET no treino (%):')
print(
    (y_train.value_counts(normalize=True) * 100)
    .round(2)
)

print('\nTARGET no teste (%):')
print(
    (y_test.value_counts(normalize=True) * 100)
    .round(2)
)

## 3. Pipeline de pré-processamento

O pré-processamento será incorporado diretamente aos pipelines de modelagem.

As variáveis numéricas receberão imputação pela mediana, indicador de ausência e padronização. As variáveis categóricas serão transformadas por *One-Hot Encoding*, enquanto as variáveis binárias já codificadas como 0 e 1 serão preservadas.

Ao integrar essas transformações ao pipeline dos modelos, cada ajuste realizado durante a validação cruzada aprende os parâmetros do pré-processamento exclusivamente a partir da respectiva parcela de treinamento, reduzindo o risco de *data leakage*.

In [ ]:
numeric_features = [
    'CNT_CHILDREN',
    'AMT_INCOME_TOTAL',
    'CNT_FAM_MEMBERS',
    'AGE_YEARS',
    'YEARS_EMPLOYED'
]

binary_features = [
    'FLAG_WORK_PHONE',
    'FLAG_PHONE',
    'FLAG_EMAIL'
]

categorical_features = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE'
]

numeric_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(
                strategy='median',
                add_indicator=True
            )
        ),
        (
            'scaler',
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore'
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)

print('Pré-processador definido com sucesso.')
print(
    f'Total de variáveis de entrada: '
    f'{len(numeric_features) + len(binary_features) + len(categorical_features)}'
)

## 4. Estratégia de avaliação dos modelos

Os modelos serão comparados por meio de **validação cruzada estratificada com 5 folds** sobre o conjunto de treino.

A estratificação preserva aproximadamente a proporção entre as classes em cada divisão. O embaralhamento dos registros será realizado com semente fixa para garantir reprodutibilidade.

Como apenas 11,77% dos clientes pertencem à classe positiva (`TARGET = 1`), a acurácia isoladamente pode produzir uma percepção inadequada do desempenho. Um modelo que previsse quase todos os clientes como bons pagadores já obteria acurácia elevada.

Por esse motivo, serão utilizadas como principais métricas de comparação:

- **ROC-AUC:** avalia a capacidade do modelo de ordenar observações positivas acima das negativas considerando diferentes limiares de classificação;
- **PR-AUC (Average Precision):** resume a relação entre precision e recall da classe positiva e recebe atenção especial neste problema devido ao desbalanceamento das classes.

Precision, recall, F1-score e matriz de confusão serão utilizados posteriormente para interpretar o comportamento das classificações produzidas pelo modelo selecionado.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision'
}

print('Validação cruzada: StratifiedKFold com 5 folds')
print('Métricas principais: ROC-AUC e PR-AUC (Average Precision)')

## 5. Regressão Logística

A Regressão Logística será utilizada como primeiro modelo de referência (*baseline*).

Inicialmente será avaliada a configuração padrão, sem ponderação das classes. Em seguida, será avaliada uma segunda versão com `class_weight='balanced'`, que atribui maior peso aos erros cometidos na classe minoritária.

A comparação permitirá verificar se o balanceamento melhora a capacidade discriminativa do modelo sem utilizar o conjunto de teste.

O pré-processamento será incorporado ao pipeline, garantindo que imputação, padronização e codificação sejam ajustadas separadamente dentro de cada fold da validação cruzada.

In [ ]:
logistic_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

cv_logistic = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print('Regressão Logística — validação cruzada')

print(
    f"ROC-AUC: "
    f"{cv_logistic['test_roc_auc'].mean():.4f} "
    f"± {cv_logistic['test_roc_auc'].std():.4f}"
)

print(
    f"PR-AUC:  "
    f"{cv_logistic['test_pr_auc'].mean():.4f} "
    f"± {cv_logistic['test_pr_auc'].std():.4f}"
)

In [ ]:
logistic_balanced_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            LogisticRegression(
                class_weight='balanced',
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

cv_logistic_balanced = cross_validate(
    logistic_balanced_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print('Regressão Logística balanceada — validação cruzada')

print(
    f"ROC-AUC: "
    f"{cv_logistic_balanced['test_roc_auc'].mean():.4f} "
    f"± {cv_logistic_balanced['test_roc_auc'].std():.4f}"
)

print(
    f"PR-AUC:  "
    f"{cv_logistic_balanced['test_pr_auc'].mean():.4f} "
    f"± {cv_logistic_balanced['test_pr_auc'].std():.4f}"
)

In [ ]:
comparacao_logistica = pd.DataFrame({
    'Modelo': [
        'Regressão Logística',
        'Regressão Logística balanceada'
    ],
    'ROC-AUC médio': [
        cv_logistic['test_roc_auc'].mean(),
        cv_logistic_balanced['test_roc_auc'].mean()
    ],
    'Desvio ROC-AUC': [
        cv_logistic['test_roc_auc'].std(),
        cv_logistic_balanced['test_roc_auc'].std()
    ],
    'PR-AUC médio': [
        cv_logistic['test_pr_auc'].mean(),
        cv_logistic_balanced['test_pr_auc'].mean()
    ],
    'Desvio PR-AUC': [
        cv_logistic['test_pr_auc'].std(),
        cv_logistic_balanced['test_pr_auc'].std()
    ]
})

comparacao_logistica.round(4)

### Resultado da Regressão Logística

A Regressão Logística apresentou desempenho discriminativo limitado. Na validação cruzada, o modelo padrão obteve **ROC-AUC médio de 0,5432** e **PR-AUC médio de 0,1476**.

A utilização de `class_weight='balanced'` não produziu melhora nessas métricas, resultando em ROC-AUC médio de **0,5425** e PR-AUC médio de **0,1455**.

Os resultados indicam que as relações capturadas pela Regressão Logística não são suficientes para separar adequadamente as classes neste conjunto de dados. Dessa forma, os próximos experimentos avaliarão modelos capazes de representar relações não lineares e interações entre as variáveis.

## 6. Random Forest

O segundo algoritmo avaliado será o **Random Forest**, um método baseado na combinação de múltiplas árvores de decisão.

Diferentemente da Regressão Logística, o Random Forest é capaz de representar relações não lineares e interações entre as características sem que essas relações precisem ser especificadas previamente.

Para esse modelo será utilizado um pré-processador próprio. As variáveis numéricas continuarão recebendo imputação pela mediana e indicador de ausência, porém **não será aplicada padronização**, pois modelos baseados em árvores não dependem da escala das variáveis.

A avaliação seguirá a mesma estratégia de validação cruzada estratificada e utilizará as mesmas métricas, permitindo comparação direta com os modelos anteriores.

In [ ]:
numeric_transformer_tree = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(
                strategy='median',
                add_indicator=True
            )
        )
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_tree, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_tree),
        (
            'model',
            RandomForestClassifier(
                n_estimators=300,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

print('Pipeline do Random Forest definido com sucesso.')

In [ ]:
cv_rf = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print('Random Forest — validação cruzada')

print(
    f"ROC-AUC: "
    f"{cv_rf['test_roc_auc'].mean():.4f} "
    f"± {cv_rf['test_roc_auc'].std():.4f}"
)

print(
    f"PR-AUC:  "
    f"{cv_rf['test_pr_auc'].mean():.4f} "
    f"± {cv_rf['test_pr_auc'].std():.4f}"
)

print('\nROC-AUC por fold:')
print(np.round(cv_rf['test_roc_auc'], 4))

print('\nPR-AUC por fold:')
print(np.round(cv_rf['test_pr_auc'], 4))

### Diagnóstico de overfitting

O Random Forest apresentou desempenho substancialmente superior ao observado nas versões da Regressão Logística durante a validação cruzada.

Entretanto, modelos baseados em árvores podem se ajustar excessivamente aos dados de treinamento. Para investigar esse comportamento, o pipeline será ajustado sobre todo o conjunto de treino e seu desempenho nesse mesmo conjunto será comparado aos resultados obtidos na validação cruzada.

Uma diferença elevada entre o desempenho no treino e na validação cruzada constitui evidência de sobreajuste (*overfitting*).

Essa análise utiliza exclusivamente o conjunto de treino. O conjunto de teste continua reservado para a avaliação final.

In [ ]:
rf_pipeline.fit(X_train, y_train)

prob_train_rf = rf_pipeline.predict_proba(X_train)[:, 1]

roc_train_rf = roc_auc_score(
    y_train,
    prob_train_rf
)

pr_train_rf = average_precision_score(
    y_train,
    prob_train_rf
)

roc_cv_rf = cv_rf['test_roc_auc'].mean()
pr_cv_rf = cv_rf['test_pr_auc'].mean()

print('Random Forest — diagnóstico de overfitting')

print(f'\nROC-AUC treino: {roc_train_rf:.4f}')
print(f'ROC-AUC CV:     {roc_cv_rf:.4f}')
print(f'Diferença:      {roc_train_rf - roc_cv_rf:.4f}')

print(f'\nPR-AUC treino:  {pr_train_rf:.4f}')
print(f'PR-AUC CV:      {pr_cv_rf:.4f}')
print(f'Diferença:      {pr_train_rf - pr_cv_rf:.4f}')

### Ajuste de hiperparâmetros do Random Forest

O diagnóstico anterior mostrou uma diferença relevante entre o desempenho do Random Forest no conjunto de treino e na validação cruzada, indicando sobreajuste.

Para investigar se uma configuração mais regularizada melhora a capacidade de generalização, será realizada uma busca em grade (*Grid Search*) sobre alguns dos principais hiperparâmetros do algoritmo.

Serão avaliadas diferentes profundidades máximas das árvores, quantidades mínimas de observações por folha, proporções de variáveis consideradas em cada divisão e estratégias de ponderação das classes.

Como a classe positiva é minoritária, a **PR-AUC (Average Precision)** será utilizada como métrica de otimização. A busca será realizada exclusivamente sobre o conjunto de treino, mantendo o conjunto de teste reservado para a avaliação final.

In [ ]:
param_grid_rf = {
    'model__max_depth': [5, 10, 15, None],
    'model__min_samples_leaf': [1, 5, 10, 20],
    'model__max_features': ['sqrt', 0.5],
    'model__class_weight': [None, 'balanced']
}

grid_rf = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid_rf,
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print('\nMelhores hiperparâmetros:')
print(grid_rf.best_params_)

print(
    f'\nMelhor PR-AUC médio: '
    f'{grid_rf.best_score_:.4f}'
)

print(
    f'PR-AUC Random Forest original: '
    f'{cv_rf["test_pr_auc"].mean():.4f}'
)

print(
    f'Diferença: '
    f'{grid_rf.best_score_ - cv_rf["test_pr_auc"].mean():.4f}'
)

### Resultado do ajuste de hiperparâmetros

A busca em grade avaliou **64 combinações de hiperparâmetros**, totalizando **320 ajustes** durante a validação cruzada.

A melhor configuração encontrada utilizou `class_weight=None`, `max_depth=None`, `max_features=0.5` e `min_samples_leaf=1`, alcançando **PR-AUC médio de 0,3988**.

O Random Forest original havia obtido PR-AUC médio de **0,3965**, representando uma diferença de apenas **0,0022**. Além disso, a melhor configuração encontrada não impôs limitações de profundidade ou quantidade mínima adicional de observações por folha.

Assim, as configurações regularizadas avaliadas não produziram melhora relevante de generalização segundo a métrica principal. Diante do ganho marginal obtido pela busca, será mantida a configuração original do Random Forest nas comparações seguintes, privilegiando uma especificação mais simples e previamente definida.

O sobreajuste identificado permanece como uma limitação do modelo e será considerado na interpretação dos resultados.

## 7. HistGradientBoosting

O terceiro algoritmo avaliado será o **HistGradientBoosting**, um método de *boosting* baseado em árvores de decisão.

Diferentemente do Random Forest, que combina diversas árvores construídas de forma independente, métodos de *boosting* constroem os estimadores sequencialmente, buscando corrigir os erros cometidos pelos modelos anteriores.

O algoritmo permite representar relações não lineares e interações entre as variáveis, oferecendo uma abordagem alternativa aos modelos avaliados anteriormente.

Será utilizada a mesma estratégia de validação cruzada estratificada com 5 folds e as mesmas métricas de avaliação, ROC-AUC e PR-AUC, permitindo comparação direta entre os algoritmos.

Assim como nas etapas anteriores, a avaliação será realizada exclusivamente sobre o conjunto de treino, mantendo o conjunto de teste reservado.

In [ ]:
categorical_transformer_hgb = Pipeline(
    steps=[
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            )
        )
    ]
)

preprocessor_hgb = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_tree, numeric_features),
        ('cat', categorical_transformer_hgb, categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)

hgb_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_hgb),
        (
            'model',
            HistGradientBoostingClassifier(
                random_state=RANDOM_STATE
            )
        )
    ]
)

print('Pipeline do HistGradientBoosting definido com sucesso.')

In [ ]:
cv_hgb = cross_validate(
    hgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print('HistGradientBoosting — validação cruzada')

print(
    f"ROC-AUC: "
    f"{cv_hgb['test_roc_auc'].mean():.4f} "
    f"± {cv_hgb['test_roc_auc'].std():.4f}"
)

print(
    f"PR-AUC:  "
    f"{cv_hgb['test_pr_auc'].mean():.4f} "
    f"± {cv_hgb['test_pr_auc'].std():.4f}"
)

print('\nROC-AUC por fold:')
print(np.round(cv_hgb['test_roc_auc'], 4))

print('\nPR-AUC por fold:')
print(np.round(cv_hgb['test_pr_auc'], 4))

## 8. Comparação e seleção do modelo

Os algoritmos avaliados serão comparados utilizando exclusivamente os resultados da validação cruzada sobre o conjunto de treino.

A seleção considerará principalmente a **PR-AUC**, por sua relevância em problemas com classe positiva minoritária, utilizando a ROC-AUC como métrica complementar. Também será observada a variação dos resultados entre os folds como indicação da estabilidade dos modelos.

A configuração ajustada do Random Forest não será incluída como um modelo adicional, pois a busca de hiperparâmetros produziu ganho marginal de PR-AUC e não solucionou o sobreajuste identificado. Dessa forma, será mantida a configuração original para a comparação final entre os algoritmos.

In [ ]:
comparacao_modelos = pd.DataFrame({
    'Modelo': [
        'Regressão Logística',
        'Regressão Logística balanceada',
        'HistGradientBoosting',
        'Random Forest'
    ],

    'ROC-AUC médio': [
        cv_logistic['test_roc_auc'].mean(),
        cv_logistic_balanced['test_roc_auc'].mean(),
        cv_hgb['test_roc_auc'].mean(),
        cv_rf['test_roc_auc'].mean()
    ],

    'Desvio ROC-AUC': [
        cv_logistic['test_roc_auc'].std(),
        cv_logistic_balanced['test_roc_auc'].std(),
        cv_hgb['test_roc_auc'].std(),
        cv_rf['test_roc_auc'].std()
    ],

    'PR-AUC médio': [
        cv_logistic['test_pr_auc'].mean(),
        cv_logistic_balanced['test_pr_auc'].mean(),
        cv_hgb['test_pr_auc'].mean(),
        cv_rf['test_pr_auc'].mean()
    ],

    'Desvio PR-AUC': [
        cv_logistic['test_pr_auc'].std(),
        cv_logistic_balanced['test_pr_auc'].std(),
        cv_hgb['test_pr_auc'].std(),
        cv_rf['test_pr_auc'].std()
    ]
})

comparacao_modelos = (
    comparacao_modelos
    .sort_values('PR-AUC médio', ascending=False)
    .reset_index(drop=True)
)

comparacao_modelos.round(4)

### Modelo selecionado

Entre os algoritmos avaliados, o **Random Forest** apresentou o melhor desempenho na validação cruzada, com **ROC-AUC médio de 0,7777** e **PR-AUC médio de 0,3965**.

O HistGradientBoosting apresentou desempenho intermediário, enquanto as duas versões da Regressão Logística demonstraram capacidade discriminativa substancialmente inferior.

Embora o Random Forest tenha apresentado evidência de sobreajuste, seu desempenho na validação cruzada permaneceu superior ao dos demais algoritmos avaliados. A busca de hiperparâmetros não encontrou uma configuração regularizada com melhora relevante na métrica principal.

Dessa forma, o Random Forest original será adotado como **modelo selecionado entre as alternativas avaliadas** e seguirá para a avaliação final no conjunto de teste, que permaneceu reservado durante todo o processo de comparação e seleção.